# 02 · Target analysis & leakage checks

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[1] if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
RAW = ROOT / "data/raw/home-credit"

from sklearn.feature_selection import mutual_info_classif
from risklens.features.registry import prepare_credit_input

In [ ]:
app = pd.read_csv(RAW / "application_train.csv")
X = prepare_credit_input(app)
num = X.select_dtypes("number").drop(columns=["TARGET", "SK_ID_CURR"])

## Correlation with the target (engineered features included)

In [ ]:
corr = num.corrwith(app.TARGET).dropna()
corr.abs().sort_values(ascending=False).head(20)

## Mutual information (non-linear dependence)

In [ ]:
sample = num.sample(30_000, random_state=0)
mi = mutual_info_classif(sample.fillna(sample.median()), app.TARGET.loc[sample.index], random_state=0)
pd.Series(mi, index=sample.columns).sort_values(ascending=False).head(20)

## Default rate by external score quintile

In [ ]:
for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    print(c, app.TARGET.groupby(pd.qcut(app[c], 5, labels=False)).mean().round(3).tolist())

## Segments (fairness preview)
Age and gender are strongly associated with default; gender is excluded from model inputs, both are audited in `fairness/report.py`.

In [ ]:
print(app.groupby("CODE_GENDER").TARGET.agg(["mean", "size"]))
print(app.TARGET.groupby(X.AGE_GROUP).agg(["mean", "size"]))

## Leakage checks
* no feature uses information after the application date (history tables are all `DAYS_* <= 0`)
* target encoding is **out-of-fold** (`features/encoding.py`) — verified by `tests/test_encoding_selection.py`
* `SK_ID_CURR` is not a feature; it is only used for the ordered (out-of-time style) robustness split

In [ ]:
b = pd.read_csv(RAW / "bureau.csv", usecols=["DAYS_CREDIT", "DAYS_CREDIT_UPDATE"])
(b.DAYS_CREDIT > 0).sum(), (b.DAYS_CREDIT_UPDATE > 0).sum()